In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/paulajibola/consumer-finance-data/SCFP2019.csv


In [2]:
import pandas as pd

import pickle
from scipy.stats.mstats import trimmed_var
import plotly.express as px
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.validation import check_is_fitted

In [3]:
!pip install scipy

In [4]:
def wrangle(filepath):
    # read into the dataframe
    df = pd.read_csv(filepath)
    mask = (df["TURNFEAR"] == 1) & (df["NETWORTH"] < 2e6)
    df = df[mask]
    return df

In [5]:
df = wrangle("/kaggle/input/datasets/paulajibola/consumer-finance-data/SCFP2019.csv")

print("df type:", type(df))
print("df shape:", df.shape)

df.head()

df type: <class 'pandas.core.frame.DataFrame'>
df shape: (4418, 351)


,YY1,Y1,WGT,HHSEX,AGE,AGECL,EDUC,EDCL,MARRIED,KIDS,...,NWCAT,INCCAT,ASSETCAT,NINCCAT,NINC2CAT,NWPCTLECAT,INCPCTLECAT,NINCPCTLECAT,INCQRTCAT,NINCQRTCAT
5,2,21,3790.476607,1,50,3,8,2,1,3,...,1,2,1,2,1,1,4,4,2,2
6,2,22,3798.868505,1,50,3,8,2,1,3,...,1,2,1,2,1,1,4,3,2,2
7,2,23,3799.468393,1,50,3,8,2,1,3,...,1,2,1,2,1,1,4,4,2,2
8,2,24,3788.076005,1,50,3,8,2,1,3,...,1,2,1,2,1,1,4,4,2,2
9,2,25,3793.066589,1,50,3,8,2,1,3,...,1,2,1,2,1,1,4,4,2,2


#### The data have so many numerical features; to determine the one to work with, we will try and find the data with the largest variance

In [6]:
# so, we use the ones with the top_ten_var

top_ten_var = df.var().sort_values().tail(10)

print("type of top ten var:", type(top_ten_var))
print("shape of top ten var:", top_ten_var.shape)
top_ten_var

type of top ten var: <class 'pandas.core.series.Series'>
shape of top ten var: (10,)


PLOAN1      1.140894e+10
ACTBUS      1.251892e+10
BUS         1.256643e+10
KGTOTAL     1.346475e+10
DEBT        1.848252e+10
NHNFIN      2.254163e+10
HOUSES      2.388459e+10
NETWORTH    4.847029e+10
NFIN        5.713939e+10
ASSET       8.303967e+10
dtype: float64

In [7]:
# create a horizontal bar chart of `top_ten_var`

fig = px.bar(
    x=top_ten_var,
    y=top_ten_var.index,
    title="Consumer Finance: Top Variance Features "
)

fig.update_layout(xaxis_title="Variance", yaxis_title="Feature")
fig.show()

In [8]:
# Many of the wealth indicators are highly skewed, with a few outliers in the enormously wealthy houses
# These outliers can affect the measure of variance. 

# Let's check if that is the same with one of the top_five_var

# create a boxplot of `NHNFIN`
fig = px.box(
    data_frame=df,
    x="NHNFIN",
    title="Distribution of Non-home, Non-Financial Assets"
)

fig.update_layout(xaxis_title="Value [$]")
fig.show()

In [9]:
df["DEBT"].var()

18482520920.408085

In [10]:
def trimmed_var(series, proportion=0.1):
    """
    Calculate sample variance after dropping a specified proportion

    from both the lower and upper tails of a pandas Series
    """
    # drop missing values
    s = series.dropna()

    # find the quantile thresholds
    low_thresh = s.quantile(proportion)
    high_thresh = s.quantile(1 - proportion)

    # keep only data points inside the bounds and calculate variance
    trimmed_data = s[(s >= low_thresh) & (s <= high_thresh)]

    return trimmed_data.var()

In [11]:
# calculate the trimmed variance which is not built into pandas
# therefore, we use the .apply() method

top_ten_trim_var = trimmed_var(df).sort_values().tail(10)


print("top_ten_var type:", type(top_ten_trim_var))
print("top_ten_var shape:", top_ten_trim_var.shape)
top_ten_trim_var

top_ten_var type: <class 'pandas.core.series.Series'>
top_ten_var shape: (10,)


WAGEINC     5.875119e+08
HOMEEQ      6.711339e+08
NH_MORT     1.231757e+09
MRTHEL      1.249892e+09
PLOAN1      1.301643e+09
DEBT        2.912459e+09
NETWORTH    3.100806e+09
HOUSES      4.576572e+09
NFIN        7.913559e+09
ASSET       1.175853e+10
dtype: float64

In [12]:
fig = px.bar(
    x=top_ten_trim_var,
    y=top_ten_trim_var.index,
    title="Consumer Finance: High Variance Features"
)

fig.update_layout(xaxis_title="Variance", yaxis_title="Feature")
fig.show()

#### Notice that the variance have described noticably after applying trimmed variance. But even with this, there is a massive gap between the `ASSET` and the `WAGEINC` features. This needs to be scaled to allow for proper cluster formation.

In [13]:
top_five_var_col = top_ten_trim_var.tail(5).index.to_list()

print("top_five_var_col type:", type(top_five_var_col))
print("top_five_var_col shape:", len(top_five_var_col))
top_five_var_col

top_five_var_col type: <class 'list'>
top_five_var_col shape: 5


['DEBT', 'NETWORTH', 'HOUSES', 'NFIN', 'ASSET']

#### SPLIT

In [14]:
# For training the model, we use 5 features [high variance features]

X = df[top_five_var_col]

print("type of X:", type(X))
print("Length of X:", len(X))
print("X shape:", X.shape)
X.head()

type of X: <class 'pandas.core.frame.DataFrame'>
Length of X: 4418
X shape: (4418, 5)


,DEBT,NETWORTH,HOUSES,NFIN,ASSET
5,12200.0,-6710.0,0.0,3900.0,5490.0
6,12600.0,-4710.0,0.0,6300.0,7890.0
7,15300.0,-8115.0,0.0,5600.0,7185.0
8,14100.0,-2510.0,0.0,10000.0,11590.0
9,15400.0,-5715.0,0.0,8100.0,9685.0


#### Iterate

It is clear from the visualization that there is a scaling issue here. We need to resolve that before we can continue

In [15]:
X_summary = X.aggregate(["mean", "std"]).astype(int)

print("type of X_summary:", type(X_summary))
print("shape of X:", X_summary.shape)
X_summary

type of X_summary: <class 'pandas.core.frame.DataFrame'>
shape of X: (2, 5)


,DEBT,NETWORTH,HOUSES,NFIN,ASSET
mean,72701,76387,74530,117330,149089
std,135950,220159,154546,239038,288166


In [16]:
# instantiate the scaler
ss = StandardScaler()

# transform
X_scaled_data = ss.fit_transform(X)

# Put `X_scaled_data` into DataFrame from nd.array
X_scaled = pd.DataFrame(X_scaled_data, columns=X.columns)

print("type x_scaled:", type(X_scaled))
print("shape x_scaled:", X_scaled.shape)
X_scaled.head()

type x_scaled: <class 'pandas.core.frame.DataFrame'>
shape x_scaled: (4418, 5)


,DEBT,NETWORTH,HOUSES,NFIN,ASSET
0,-0.445075,-0.377486,-0.48231,-0.474583,-0.498377
1,-0.442132,-0.368401,-0.48231,-0.464541,-0.490047
2,-0.422270,-0.383868,-0.48231,-0.467470,-0.492494
3,-0.431097,-0.358407,-0.48231,-0.449061,-0.477206
4,-0.421534,-0.372966,-0.48231,-0.457010,-0.483818


In [17]:
X_scaled_summary = X_scaled.aggregate(["mean", "std"]).astype(int)

print("type of x_scaled_summary:", type(X_scaled_summary))
print("shape of x_scaled_summary:", X_scaled_summary.shape)
print("mean = 0, std = 1. standard scaling successful...")
X_scaled_summary

type of x_scaled_summary: <class 'pandas.core.frame.DataFrame'>
shape of x_scaled_summary: (2, 5)
mean = 0, std = 1. standard scaling successful...


,DEBT,NETWORTH,HOUSES,NFIN,ASSET
mean,0,0,0,0,0
std,1,1,1,1,1


In [18]:
n_clusters = range(2, 13)
inertia_errors = []
silhouette_scores = []

# add `for` loop to train model and calculate inertia and silhouette scores
for k in n_clusters:
    # build the pipeline
    model = make_pipeline(StandardScaler(), KMeans(n_clusters=k, random_state=42))
    # train the model
    model.fit(X)
    
    # access the fitted KMeans Object from the pipeline
    fitted_kmeans = model.named_steps["kmeans"]
    # calculate inertia
    inertia_errors.append(fitted_kmeans.inertia_)
    
    # calculate the silhouette score
    s_score = silhouette_score(X, fitted_kmeans.labels_)
    
    silhouette_scores.append(s_score)
    
print("inertia_errors type:", type(inertia_errors))
print("inertia errors len:", len(inertia_errors))
print("inertia:", inertia_errors)
print()

print("silhouette_scores type:", type(silhouette_scores))
print("silhouette_scores len:", len(silhouette_scores))
print("silhouette_scores:", silhouette_scores)


inertia_errors type: <class 'list'>
inertia errors len: 11
inertia: [11028.058082607175, 7190.703769068748, 6233.106346879298, 5009.683910188738, 4935.578599045223, 4282.331488864099, 4103.6585674172, 3503.109897216321, 2874.9614912020043, 2646.094394137889, 2505.471308202903]

silhouette_scores type: <class 'list'>
silhouette_scores len: 11
silhouette_scores: [np.float64(0.7464502937083215), np.float64(0.7042841259119152), np.float64(0.7046977054481021), np.float64(0.6563064140490201), np.float64(0.6589817231149715), np.float64(0.6414988659227319), np.float64(0.6423235394675234), np.float64(0.6453132882450023), np.float64(0.6392046929708322), np.float64(0.6214410819897591), np.float64(0.6237975162108229)]


In [19]:
# create a plot of `inertia errors` vs `n_clusters`

fig = px.line(
    x=n_clusters, y=inertia_errors, title="K-Means Model: Inertia vs. No. of Clusters"
)

fig.update_layout(xaxis_title="No of Clusters [k]", yaxis_title="Inertia")
fig.show()

In [20]:
# create the 'silhouette_scores' vs 'n_clusters'

fig = px.line(
    x=n_clusters,
    y=silhouette_scores,
    title="K-Means Model: Silhouette Score vs. No. of Clusters"
)

fig.update_layout(xaxis_title="Number of Clusters (k)", yaxis_title="Silhouette Score")
fig.show()

#### 3 appears to be the perfect value for k

In [21]:
# build the model
final_model = make_pipeline(
    StandardScaler(),
    KMeans(n_clusters=3, random_state=42)
)

# fit the model
final_model.fit(X)

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('kmeans', KMeans(n_clusters=3, random_state=42))])

In [22]:
check_is_fitted(final_model)
print("model is readyyy...")

model is readyyy...


In [24]:
final_labels = final_model.named_steps["kmeans"].labels_

print("label types:", type(final_labels))
print("label length:", len(final_labels))
final_labels

label types: <class 'numpy.ndarray'>
label length: 4418


array([1, 1, 1, ..., 0, 0, 0], dtype=int32)

In [25]:
X.head()

,DEBT,NETWORTH,HOUSES,NFIN,ASSET
5,12200.0,-6710.0,0.0,3900.0,5490.0
6,12600.0,-4710.0,0.0,6300.0,7890.0
7,15300.0,-8115.0,0.0,5600.0,7185.0
8,14100.0,-2510.0,0.0,10000.0,11590.0
9,15400.0,-5715.0,0.0,8100.0,9685.0


In [26]:
xgb = X.groupby(final_labels).mean()
print("xgb type:", type(xgb))
print("xgb shape:", xgb.shape)
xgb

xgb type: <class 'pandas.core.frame.DataFrame'>
xgb shape: (3, 5)


,DEBT,NETWORTH,HOUSES,NFIN,ASSET
0,205890.006211,226982.254658,258129.813665,3.478201e+05,4.328723e+05
1,28566.896108,11742.759015,14600.057241,2.813948e+04,4.030966e+04
2,467562.773109,955724.773109,592184.873950,1.176909e+06,1.423288e+06


In [27]:
final_labels

array([1, 1, 1, ..., 0, 0, 0], dtype=int32)

In [28]:
fig = px.bar(
    xgb,
    barmode="group",
    title="Mean Household Finances by Cluster"
)

fig.update_layout(xaxis_title="Cluster", yaxis_title="Value [$]")

In [29]:
# Instantiate Transformer
pca = PCA(n_components=2, random_state=42)

# fit the transformer
X_t = pca.fit_transform(X)

X_t

array([[-221525.4245304 ,   22052.27300297],
       [-217775.10072188,   22851.35806823],
       [-219519.64217476,   19023.64633269],
       ...,
       [ 323306.39571853,  182741.08760824],
       [ 327948.76711284,  184322.85705512],
       [ 334191.95622931,  186450.06424231]])

In [30]:
# put in a df
X_pca = pd.DataFrame(X_t, columns=["PC1", "PC2"])

print("X_pca type:", type(X_pca))
print("X_pca shape:", X_pca.shape)
X_pca.sample(5)

X_pca type: <class 'pandas.core.frame.DataFrame'>
X_pca shape: (4418, 2)


,PC1,PC2
2685,-213701.247836,40723.945807
361,-211653.495886,42834.240805
1330,-226692.185514,34322.821008
204,48674.695883,-51678.241946
349,-216373.956470,34606.689014


In [31]:
labels_pd = pd.Series(final_labels)

In [32]:
labels_pd.max()

2

In [33]:
new_labels = labels_pd.sort_values().astype(str)    # convert the values of the labels into `str` so labels do not presebt as continuous variables, rather they present as categorical variables
new_labels

29      0
4413    0
800     0
801     0
802     0
       ..
889     2
888     2
887     2
886     2
885     2
Length: 4418, dtype: object

In [34]:
fig = px.scatter(
    data_frame=X_pca,
    x="PC1",
    y="PC2",
    color=labels_pd.astype(str),    # to convert continuous variable to categorical variable
    title="PCA Representation of Clusters"
)
fig.update_layout(xaxis_title="Value [$]", yaxis_title="Cluster")
fig.show()